# DigiPin DISHA — QLoRA Fine-Tuning on Colab

Fine-tunes **Qwen2.5-7B** (4-bit) on 1,000 urban intelligence Q&A pairs using **Unsloth + QLoRA**.

**Requirements:** Colab T4 GPU (free tier works) — ~15 min training time.

**Pipeline:**
1. Install dependencies
2. Upload training data (`digipin-alpaca.json`)
3. Load 4-bit model + attach LoRA adapters
4. Train (3 epochs, ~15 min on T4)
5. Test inference
6. Export to GGUF for Ollama
7. Download artifacts

## 0. GPU Check

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime > Change runtime type > T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
print("Dependencies installed.")

## 2. Upload Training Data

Upload your `digipin-alpaca.json` file (generated from the DigiPin portal).

**Option A:** Run the cell below to upload interactively.  
**Option B:** Place the file in your Google Drive and mount it.

In [ ]:
import os
import json

DATA_PATH = "digipin-alpaca.json"

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Upload digipin-alpaca.json:")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

with open(DATA_PATH, "r") as f:
    data = json.load(f)
print(f"Loaded {len(data)} training pairs")
print(f"Sample instruction: {data[0]['instruction'][:80]}...")

## 3. Load Model (4-bit Quantized)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-bnb-4bit"
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print(f"Model loaded: {MODEL_NAME}")

## 4. Attach LoRA Adapters

In [ ]:
LORA_R = 16
LORA_ALPHA = 16

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. Prepare Dataset

In [ ]:
from datasets import Dataset

ALPACA_TEMPLATE = """Below is an instruction that describes an urban analysis task, paired with context data from India's DigiPin system. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_prompt(example):
    return {"text": ALPACA_TEMPLATE.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"],
    )}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

print(f"Dataset: {len(dataset)} examples")
print(f"\nSample (first 300 chars):\n{dataset[0]['text'][:300]}...")

## 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "disha-qlora-output"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Runtime: {stats.metrics['train_runtime']:.0f}s")
print(f"  Samples/sec: {stats.metrics['train_samples_per_second']:.1f}")

## 7. Test Inference

Quick sanity check — ask the fine-tuned model a question with sample context.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Below is an instruction that describes an urban analysis task, paired with context data from India's DigiPin system. Write a response that appropriately completes the request.

### Instruction:
Give me a full urban intelligence briefing for this location.

### Input:
=== LOCATION: DigiPin 3MC-99P-ABCD ===
Coordinates: 22.7196N, 75.8577E
Full: Vijay Nagar, Indore, Madhya Pradesh, India
Weather: Temp: 32C, Humidity: 55%, Wind: 8km/h, Partly cloudy
Air Quality: AQI=120 (Moderate), PM2.5=45, PM10=62
Solar: GHI=5.8 kWh/m2/day, Sunshine=9h, Potential: Good
Elevation: 553m (+3m vs surroundings)

=== INTELLIGENCE SCORES (0-100) ===
livability=68, walkability=72, safety=45, green=30, healthcare=55, education=61, commercial=78, connectivity=82, noise_estimate=35, flood_risk=20

### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1,
)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("MODEL RESPONSE:")
print("=" * 60)
print(response)

## 8. Save LoRA Adapter

In [ ]:
LORA_DIR = f"{OUTPUT_DIR}/lora-adapter"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA adapter saved to: {LORA_DIR}")

# Show adapter size
import subprocess
result = subprocess.run(["du", "-sh", LORA_DIR], capture_output=True, text=True)
print(f"Adapter size: {result.stdout.strip()}")

## 9. Export to GGUF (for Ollama)

Converts the fine-tuned model to **Q4_K_M** GGUF format for local inference with Ollama.

In [ ]:
GGUF_DIR = f"{OUTPUT_DIR}/gguf"

print("Exporting to GGUF Q4_K_M (this takes a few minutes)...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF exported to: {GGUF_DIR}")

# Show file size
import glob
for f in glob.glob(f"{GGUF_DIR}/*.gguf"):
    size_gb = os.path.getsize(f) / 1e9
    print(f"  {os.path.basename(f)}: {size_gb:.2f} GB")

## 10. Download Artifacts

Download the GGUF file and LoRA adapter to your local machine.

In [ ]:
from google.colab import files
import glob as g

# Download GGUF
gguf_files = g.glob(f"{GGUF_DIR}/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("No GGUF file found. Run the export cell first.")

## 11. (Optional) Push to Hugging Face Hub

In [ ]:
# Uncomment and fill in to push to HF Hub:

# HF_REPO = "your-username/disha-qwen2.5-7b-qlora"
# HF_TOKEN = "hf_..."
#
# model.push_to_hub(HF_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
# print(f"Pushed to: https://huggingface.co/{HF_REPO}")

## Local Setup (after download)

Once you've downloaded the GGUF file:

```bash
# 1. Update the Modelfile to point to your GGUF
#    Change FROM line to: FROM ./path/to/unsloth.Q4_K_M.gguf

# 2. Create the Ollama model
ollama create disha -f Modelfile

# 3. Test it
ollama run disha
```